# 04 · Decision Framework

Action: Convert scores to simple decisions we can act on.
So what: Clear shortlist enables focused scouting and partnership moves.


In [ ]:
# Notebook bootstrap: ensure project root is on sys.path for `import src.*`
import os, sys
from pathlib import Path
nb_cwd = Path.cwd()
root = None
for p in [nb_cwd, *nb_cwd.parents]:
    if (p / 'pyproject.toml').exists() or (p / 'src').exists():
        root = p; break
if root and str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.environ['PYTHONPATH'] = str(root) + (':' + os.environ.get('PYTHONPATH','') if os.environ.get('PYTHONPATH') else '')
print(f'Project root: {root}')


In [ ]:
import pandas as pd
from src.config import load_settings
from src import qa as _qa
from src.storytelling import so_what

settings = load_settings()
with _qa.time_limit(settings.execution_timeout_s):
    scores = pd.read_csv(settings.processed_dir / 'artist_breakout_scores.csv', parse_dates=['last_metrics_date'])

summary = (scores.groupby('decision')
           .agg(artists=('artist_name','nunique'),
                avg_score=('composite_score','mean'))
           .sort_values('avg_score', ascending=False))
display(summary)
so_what("Shortlist concentrates top momentum — redirect this week's discovery time accordingly.")

shortlist = scores[scores['decision'] == 'breakout_candidate'].sort_values('composite_score', ascending=False)
display(shortlist[['artist_name','composite_score','view_momentum','engagement_velocity','fanbase_acceleration']].head(10))

# Export decisions for stakeholders
out_csv = settings.export_dir / 'breakout_decisions.csv'
scores.to_csv(out_csv, index=False)
print({'exported': str(out_csv), 'rows': len(scores)})


In [ ]:
# Quantified impact (proxy)
flagged_count = int((scores['decision'] == 'breakout_candidate').sum())
reviewed_count = max(flagged_count, 1)  # assume we review flagged artists
baseline_discovery_rate = 0.20  # proxy from impact/impact_notes.md
model_precision = 0.50  # central estimate; refine with backtests
evaluation_window = '90-day window'

if flagged_count <= 0:
    raise RuntimeError('Impact cannot be quantified: no breakout candidates flagged.')

baseline_hits = baseline_discovery_rate * reviewed_count
model_hits = model_precision * flagged_count

if baseline_hits <= 0:
    raise RuntimeError('Invalid baseline; cannot compute lift.')

lift_ratio = model_hits / baseline_hits
lift_delta_pp = (model_precision - baseline_discovery_rate) * 100.0

impact_table = pd.DataFrame([
    {'metric': 'discovery_rate', 'baseline': baseline_discovery_rate, 'model': model_precision, 'window': evaluation_window},
    {'metric': 'expected_hits', 'baseline': baseline_hits, 'model': model_hits, 'window': evaluation_window},
])
display(impact_table)

print({
    'flagged': flagged_count,
    'reviewed': reviewed_count,
    'baseline_discovery_rate': baseline_discovery_rate,
    'model_precision': model_precision,
    'lift_ratio': round(lift_ratio, 2),
    'lift_delta_percentage_points': round(lift_delta_pp, 1),
    'evaluation_window': evaluation_window
})
so_what(f'Projected discovery efficiency +{lift_delta_pp:.1f}pp vs baseline (x{lift_ratio:.2f} lift) — {evaluation_window}.')
